# 🐍 MAMBA MODELS — Run All (Classification + Regression)

All Mamba variants on the same eye-image → HB dataset.  
**Edit only Section 1. Nothing else needs to change.**

| # | Model | Paper |
|---|-------|-------|
| 1 | Mamba1 (official SSM) | Gu & Dao 2023 |
| 2 | Mamba2 (SSD) | Dao & Gu ICML 2024 |
| 3 | Mamba3 (MIMO + Rotary) | ICLR 2026 |
| 4 | VMamba (2D cross-scan) | ICLR 2025 |
| 5 | MambaVision (Mamba+Transformer) | NVIDIA CVPR 2025 |
| 6 | MedMamba (medical imaging) | 2024 |
| 7 | DSA-Mamba (official, First-Ronin) | 2024 |


## ★ Section 1 — Configuration  *(Edit only here)*

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                  ★  ALL VARIABLES — EDIT HERE  ★                   ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── TASK ─────────────────────────────────────────────────────────────────────
# Choose what each model should do:
#   "classification" → predict Anemic vs Non-Anemic (binary, CrossEntropy)
#   "regression"     → predict exact HB value in g/dL (MSE)
#   "both"           → classification AND regression simultaneously (default)

TASK = "both"


# ── DATASET PATHS ─────────────────────────────────────────────────────────────
IMAGE_DIR  = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye"
CSV_PATH   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx"
IMAGE_COL  = "Patient ID"   # column whose value is the image filename (no extension)
HB_COL     = "HB"           # column with the HB measurement
HB_THRESH  = 12.0           # g/dL — below this → anemic (label 0), above → normal (label 1)


# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS      = 2             # epochs per model (increase for real training)
BATCH_SIZE  = 4
LR          = 1e-4
VAL_SPLIT   = 0.2
NUM_WORKERS = 2
SEED        = 42

# Loss weights (used only when TASK = "both")
CLS_WEIGHT  = 1.0           # weight for CrossEntropy classification loss
REG_WEIGHT  = 0.5           # weight for MSE regression loss


# ── IMAGE ─────────────────────────────────────────────────────────────────────
IMG_SIZE    = 224


# ── WHICH MODELS TO RUN ───────────────────────────────────────────────────────
# Set False to skip a model (e.g. if a kernel fails to install)
RUN = {
    "Mamba1"      : True,
    "Mamba2"      : True,
    "Mamba3"      : True,
    "VMamba"      : True,
    "MambaVision" : True,
    "MedMamba"    : True,
    "DSA-Mamba"   : True,
}


# ── REPO ROOT ─────────────────────────────────────────────────────────────────
# After:  git clone https://github.com/<you>/MAMBA_MODELS.git
# Kaggle working dir is /kaggle/working — repo clones into /kaggle/working/MAMBA_MODELS
import os
REPO_ROOT = os.path.dirname(os.path.abspath("__file__"))   # auto-detected
# If auto-detection is wrong, set manually:
# REPO_ROOT = "/kaggle/working/MAMBA_MODELS"

print(f"TASK        : {TASK}")
print(f"EPOCHS      : {EPOCHS}")
print(f"BATCH_SIZE  : {BATCH_SIZE}")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"Models ON   : {[k for k,v in RUN.items() if v]}")


## Section 2 — Install Dependencies

In [ ]:
import subprocess, sys

PKGS = [
    "mamba-ssm",        # official compiled Mamba (needs CUDA GPU on Kaggle)
    "causal-conv1d",    # required by mamba-ssm
    "mambavision",      # NVIDIA MambaVision (pip package)
    "einops",
    "timm",
    "pandas",
    "openpyxl",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]

for pkg in PKGS:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True)
    status = "✅" if result.returncode == 0 else "⚠️ "
    print(f"  {status} {pkg}")

print("\nDone.")


## Section 3 — Shared Imports & Dataset

In [ ]:
import os, sys, math, time, warnings, traceback
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              roc_auc_score, mean_absolute_error, f1_score)
from einops import rearrange, repeat
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Task   : {TASK}")


In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class EyeHBDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        pid   = str(row[IMAGE_COL])
        hb    = float(row[HB_COL])
        label = 0 if hb < HB_THRESH else 1    # 0=anemic, 1=normal

        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ""]:
            p = os.path.join(IMAGE_DIR, pid + ext)
            if os.path.exists(p):
                img_path = p; break
        if img_path is None:
            raise FileNotFoundError(f"No image for ID '{pid}' in {IMAGE_DIR}")

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return (img,
                torch.tensor(label,  dtype=torch.long),
                torch.tensor([[hb]], dtype=torch.float32))


T_TRAIN = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
T_VAL = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])


def load_data():
    if CSV_PATH.endswith((".xlsx", ".xls")):
        df = pd.read_excel(CSV_PATH)
    else:
        df = pd.read_csv(CSV_PATH)

    labels_bin = (df[HB_COL] < HB_THRESH).astype(int)
    train_df, val_df = train_test_split(
        df, test_size=VAL_SPLIT, random_state=SEED, stratify=labels_bin)

    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
               pin_memory=(DEVICE == "cuda"))
    train_loader = DataLoader(EyeHBDataset(train_df, T_TRAIN), shuffle=True,  **kw)
    val_loader   = DataLoader(EyeHBDataset(val_df,   T_VAL),   shuffle=False, **kw)

    print(f"Samples  → total:{len(df)}  train:{len(train_df)}  val:{len(val_df)}")
    print(f"Anemic   → {labels_bin.sum()}/{len(df)}  ({labels_bin.mean()*100:.1f}%)")
    print(f"HB stats → mean:{df[HB_COL].mean():.1f}  "
          f"min:{df[HB_COL].min():.1f}  max:{df[HB_COL].max():.1f}")
    return train_loader, val_loader


TRAIN_LOADER, VAL_LOADER = load_data()


## Section 4 — Shared Training Utilities

In [ ]:
# ── Loss function (adapts to TASK) ────────────────────────────────────────────
CE_FN  = nn.CrossEntropyLoss()
MSE_FN = nn.MSELoss()

def compute_loss(logits, hb_pred, labels, hb_true):
    """Returns total loss depending on global TASK variable."""
    if TASK == "classification":
        return CE_FN(logits, labels), CE_FN(logits, labels).item(), 0.0
    elif TASK == "regression":
        return MSE_FN(hb_pred, hb_true), 0.0, MSE_FN(hb_pred, hb_true).item()
    else:  # "both"
        cl = CE_FN(logits, labels)
        rl = MSE_FN(hb_pred, hb_true)
        return CLS_WEIGHT * cl + REG_WEIGHT * rl, cl.item(), rl.item()


# ── Single epoch (train or eval) ──────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_preds, all_labels, all_scores = [], [], []
    all_hbp,   all_hbt               = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        bar = tqdm(loader, leave=False,
                   desc="  train" if training else "  val  ")
        for imgs, labels, hb_true in bar:
            imgs, labels, hb_true = (imgs.to(DEVICE),
                                     labels.to(DEVICE),
                                     hb_true.to(DEVICE))
            logits, hb_pred = model(imgs)
            loss, cl, rl    = compute_loss(logits, hb_pred, labels, hb_true)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()

            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_scores.extend(probs[:, 1].cpu().tolist())
            all_hbp.extend(hb_pred.detach().cpu().squeeze().tolist())
            all_hbt.extend(hb_true.cpu().squeeze().tolist())

    n    = len(loader)
    metrics = {"loss": total_loss / n}

    if TASK in ("classification", "both"):
        metrics["acc"] = accuracy_score(all_labels, all_preds)
        metrics["f1"]  = f1_score(all_labels, all_preds, average="macro",
                                   zero_division=0)
        try:
            metrics["auc"] = roc_auc_score(all_labels, all_scores)
        except Exception:
            metrics["auc"] = float("nan")

    if TASK in ("regression", "both"):
        arr_t = np.array(all_hbt); arr_p = np.array(all_hbp)
        metrics["mae"]  = mean_absolute_error(arr_t, arr_p)
        metrics["rmse"] = float(np.sqrt(np.mean((arr_t - arr_p) ** 2)))

    return metrics, all_labels, all_preds, all_hbt, all_hbp


# ── Full training run for one model ───────────────────────────────────────────
RESULTS = []   # summary collected across all models

def run_model(name, model):
    """Train `model` for EPOCHS, log metrics, store result."""
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'═'*65}")
    print(f"  {name}   ({params/1e6:.2f}M params)   Task={TASK}")
    print(f"{'═'*65}")

    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS)

    hist = {k: [] for k in ["tr_loss", "vl_loss", "acc", "auc", "f1", "mae", "rmse"]}
    t0   = time.time()

    for ep in range(1, EPOCHS + 1):
        tr, *_ = run_epoch(model, TRAIN_LOADER, optimizer)
        vl, vl_labels, vl_preds, vl_hbt, vl_hbp = run_epoch(model, VAL_LOADER)
        scheduler.step()

        hist["tr_loss"].append(tr["loss"])
        hist["vl_loss"].append(vl["loss"])
        for k in ["acc", "auc", "f1", "mae", "rmse"]:
            hist[k].append(vl.get(k, float("nan")))

        # ── Per-epoch console line ─────────────────────────────────────────
        line = (f"  Ep {ep}/{EPOCHS}  "
                f"TL:{tr['loss']:.4f}  VL:{vl['loss']:.4f}")
        if TASK in ("classification", "both"):
            line += f"  Acc:{vl.get('acc',0):.3f}  AUC:{vl.get('auc',0):.3f}"
        if TASK in ("regression", "both"):
            line += f"  MAE:{vl.get('mae',0):.2f}g/dL  RMSE:{vl.get('rmse',0):.2f}"
        print(line)

    elapsed = time.time() - t0

    # ── Final classification report ────────────────────────────────────────
    if TASK in ("classification", "both"):
        print("\n  Classification Report (final epoch):")
        print(classification_report(vl_labels, vl_preds,
              target_names=["Anemic","Normal"], zero_division=0))

    # ── Store result ───────────────────────────────────────────────────────
    last = {k: (v[-1] if v else float("nan")) for k, v in hist.items()}
    RESULTS.append(dict(
        name=name, status="✅ PASS", params=params,
        time_s=elapsed, **last, error=""))

    # ── Loss curve ─────────────────────────────────────────────────────────
    ep_x = range(1, EPOCHS + 1)
    fig, axes = plt.subplots(1, 3 if TASK == "both" else 2, figsize=(14, 3))
    axes[0].plot(ep_x, hist["tr_loss"], label="Train")
    axes[0].plot(ep_x, hist["vl_loss"], label="Val")
    axes[0].set_title(f"{name} — Loss"); axes[0].legend()
    col = 1
    if TASK in ("classification", "both"):
        axes[col].plot(ep_x, hist["acc"],  label="Acc",  color="green")
        axes[col].plot(ep_x, hist["auc"],  label="AUC",  color="purple", linestyle="--")
        axes[col].plot(ep_x, hist["f1"],   label="F1",   color="teal",   linestyle=":")
        axes[col].set_title("Classification Metrics"); axes[col].legend(); axes[col].set_ylim(0,1)
        col += 1
    if TASK in ("regression", "both"):
        axes[col].plot(ep_x, hist["mae"],  label="MAE (g/dL)",  color="orange")
        axes[col].plot(ep_x, hist["rmse"], label="RMSE (g/dL)", color="red", linestyle="--")
        axes[col].set_title("HB Regression Metrics"); axes[col].legend()
    plt.suptitle(name, fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig(f"results_{name.replace(' ','_')}.png", dpi=100, bbox_inches="tight")
    plt.show()
    print(f"  ✅  Done in {elapsed:.0f}s")
    return model


## Section 5 — Pure-PyTorch SSM Base Block
*(Used by Mamba1/2/3 when the compiled kernel isn't available)*

In [ ]:
# Pure-PyTorch selective SSM — no CUDA kernel needed, runs on CPU/GPU
class _PureMamba(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        d_inner       = int(expand * d_model)
        self.d_inner  = d_inner; self.d_state = d_state
        self.in_proj  = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d   = nn.Conv1d(d_inner, d_inner, d_conv,
                                   padding=d_conv-1, groups=d_inner)
        self.act      = nn.SiLU()
        self.x_proj   = nn.Linear(d_inner, d_state*2+1, bias=False)
        self.dt_proj  = nn.Linear(1, d_inner, bias=True)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        A = repeat(torch.arange(1, d_state+1, dtype=torch.float32), "n->d n", d=d_inner)
        self.A_log = nn.Parameter(torch.log(A))
        self.D     = nn.Parameter(torch.ones(d_inner))

    def forward(self, x):                               # (B, L, d_model)
        B, L, _ = x.shape
        xz = self.in_proj(x); x_, z = xz.chunk(2, dim=-1)
        x_ = rearrange(x_, "b l d->b d l")
        x_ = self.conv1d(x_)[..., :L]
        x_ = rearrange(x_, "b d l->b l d"); x_ = self.act(x_)
        bcd = self.x_proj(x_)
        B_  = bcd[..., :self.d_state]
        C   = bcd[..., self.d_state:2*self.d_state]
        dt  = F.softplus(self.dt_proj(bcd[..., -1:]))
        A   = -torch.exp(self.A_log.float())
        dA  = torch.exp(dt.unsqueeze(-1) * A)
        dB  = dt.unsqueeze(-1) * B_.unsqueeze(2)
        h   = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys  = []
        for i in range(L):
            h = dA[:, i]*h + dB[:, i]*x_[:, i:i+1].unsqueeze(-1)
            ys.append((h * C[:, i].unsqueeze(1)).sum(-1))
        y = torch.stack(ys, dim=1) + x_ * self.D
        return self.out_proj(y * self.act(z))


# Shared vision wrapper: PatchEmbed → SSM blocks → CLS → dual head
class _VisionMambaDual(nn.Module):
    def __init__(self, ssm_fn, embed_dim=128, depth=4, patch_size=16, num_classes=2):
        super().__init__()
        n = (IMG_SIZE // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, embed_dim, patch_size, stride=patch_size)
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.zeros(1, n+1, embed_dim))
        trunc_normal_(self.cls_token, std=.02); trunc_normal_(self.pos_embed, std=.02)
        self.norms  = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(depth)])
        self.blocks = nn.ModuleList([ssm_fn(embed_dim) for _ in range(depth)])
        self.norm   = nn.LayerNorm(embed_dim)
        self.cls_head = nn.Linear(embed_dim, num_classes)
        self.reg_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim//2), nn.GELU(),
            nn.Linear(embed_dim//2, 1))

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1,2)       # (B,N,C)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], 1) + self.pos_embed
        for norm, blk in zip(self.norms, self.blocks):
            x = x + blk(norm(x))
        f = self.norm(x)[:, 0]
        return self.cls_head(f), self.reg_head(f)

print("✅ Pure-PyTorch SSM base ready")


---
## Model 1 — Mamba1 (Official SSM)
**Repo:** state-spaces/mamba &nbsp;|&nbsp; **Paper:** Gu & Dao 2023

In [ ]:
if RUN["Mamba1"]:
    try:
        from mamba_ssm import Mamba as _M1
        print("✅ compiled mamba_ssm found — using CUDA Mamba1 kernel")
        _m1_fn = lambda d: _M1(d_model=d, d_state=16, d_conv=4, expand=2)
    except ImportError:
        print("⚠️  mamba_ssm not installed — using pure-PyTorch fallback")
        _m1_fn = lambda d: _PureMamba(d, d_state=16)

    mamba1_model = _VisionMambaDual(ssm_fn=_m1_fn, embed_dim=128, depth=4)
    print(f"Mamba1 built")
else:
    print("⏭ Mamba1 skipped (RUN['Mamba1'] = False)")


In [ ]:
if RUN["Mamba1"]:
    try:
        run_model("Mamba1 (official SSM)", mamba1_model)
    except Exception as e:
        print(f"❌ Mamba1 FAILED: {e}")
        traceback.print_exc()
        RESULTS.append(dict(name="Mamba1", status="❌ FAIL", error=str(e),
            params=0, time_s=0, tr_loss=float('nan'), vl_loss=float('nan'),
            acc=float('nan'), auc=float('nan'), f1=float('nan'),
            mae=float('nan'), rmse=float('nan')))


---
## Model 2 — Mamba2 (SSD)
**Repo:** state-spaces/mamba &nbsp;|&nbsp; **Paper:** Dao & Gu ICML 2024

In [ ]:
if RUN["Mamba2"]:
    try:
        from mamba_ssm.modules.mamba2_simple import Mamba2Simple as _M2
        print("✅ compiled Mamba2Simple found")
        _m2_fn = lambda d: _M2(d_model=d, d_state=64, d_conv=4, expand=2)
    except Exception:
        print("⚠️  Mamba2 kernel not found — using pure-PyTorch (larger d_state=64)")
        _m2_fn = lambda d: _PureMamba(d, d_state=64)

    mamba2_model = _VisionMambaDual(ssm_fn=_m2_fn, embed_dim=128, depth=4)
    print("Mamba2 built")
else:
    print("⏭ Mamba2 skipped")


In [ ]:
if RUN["Mamba2"]:
    try:
        run_model("Mamba2 (SSD, ICML 2024)", mamba2_model)
    except Exception as e:
        print(f"❌ Mamba2 FAILED: {e}"); traceback.print_exc()
        RESULTS.append(dict(name="Mamba2", status="❌ FAIL", error=str(e),
            params=0, time_s=0, tr_loss=float('nan'), vl_loss=float('nan'),
            acc=float('nan'), auc=float('nan'), f1=float('nan'),
            mae=float('nan'), rmse=float('nan')))


---
## Model 3 — Mamba3 (MIMO + Rotary)
**Repo:** state-spaces/mamba &nbsp;|&nbsp; **Paper:** ICLR 2026 Oral

In [ ]:
if RUN["Mamba3"]:
    sys.path.insert(0, os.path.join(REPO_ROOT, "01_Mamba_Official"))
    try:
        from mamba_ssm.modules.mamba3 import Mamba3 as _M3
        print("✅ compiled Mamba3 found")
        _m3_fn = lambda d: _M3(d_model=d)
    except Exception:
        print("⚠️  Mamba3 not installed — using pure-PyTorch approximation")
        _m3_fn = lambda d: _PureMamba(d, d_state=32, expand=2)

    mamba3_model = _VisionMambaDual(ssm_fn=_m3_fn, embed_dim=128, depth=4)
    print("Mamba3 built")
else:
    print("⏭ Mamba3 skipped")


In [ ]:
if RUN["Mamba3"]:
    try:
        run_model("Mamba3 (ICLR 2026 Oral)", mamba3_model)
    except Exception as e:
        print(f"❌ Mamba3 FAILED: {e}"); traceback.print_exc()
        RESULTS.append(dict(name="Mamba3", status="❌ FAIL", error=str(e),
            params=0, time_s=0, tr_loss=float('nan'), vl_loss=float('nan'),
            acc=float('nan'), auc=float('nan'), f1=float('nan'),
            mae=float('nan'), rmse=float('nan')))


---
## Model 4 — VMamba (2D Cross-Scan SSM)
**Repo:** MzeroMiko/VMamba &nbsp;|&nbsp; **Paper:** ICLR 2025

In [ ]:
if RUN["VMamba"]:
    vmamba_dir = os.path.join(REPO_ROOT, "02_VMamba")
    sys.path.insert(0, vmamba_dir)
    sys.path.insert(0, os.path.join(vmamba_dir, "classification"))

    _vmamba_backbone = None
    _vmamba_feat_dim = 768

    # Attempt 1: classification/models/vmamba.py
    try:
        from models.vmamba import VSSM as _VMambaVSSM
        _vmamba_backbone = _VMambaVSSM(num_classes=0, depths=[2,2,9,2], dims=96)
        _vmamba_feat_dim = _vmamba_backbone.num_features
        print(f"✅ VMamba VSSM loaded (feat_dim={_vmamba_feat_dim})")
    except Exception as e1:
        # Attempt 2: vmamba.py at root of 02_VMamba
        try:
            import importlib.util
            spec = importlib.util.spec_from_file_location(
                "vmamba_mod", os.path.join(vmamba_dir, "vmamba.py"))
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            _vmamba_backbone = mod.VSSM(num_classes=0)
            _vmamba_feat_dim = _vmamba_backbone.num_features
            print(f"✅ VMamba (vmamba.py) loaded (feat_dim={_vmamba_feat_dim})")
        except Exception as e2:
            print(f"⚠️  VMamba import failed ({e2}) — using ResNet-18 fallback")
            from torchvision.models import resnet18
            _vmamba_backbone = resnet18(weights=None)
            _vmamba_feat_dim = 512
            _vmamba_backbone.fc = nn.Identity()

    class _VMambaDual(nn.Module):
        def __init__(self, bb, fd):
            super().__init__()
            self.bb = bb
            self.cls_head = nn.Linear(fd, 2)
            self.reg_head = nn.Sequential(
                nn.Linear(fd, fd//4), nn.GELU(), nn.Linear(fd//4, 1))
        def forward(self, x):
            f = self.bb(x)
            if f.dim() > 2: f = f.mean([-2,-1])
            return self.cls_head(f), self.reg_head(f)

    vmamba_model = _VMambaDual(_vmamba_backbone, _vmamba_feat_dim)
    print("VMamba dual-head built")
else:
    print("⏭ VMamba skipped")


In [ ]:
if RUN["VMamba"]:
    try:
        run_model("VMamba (2D cross-scan, ICLR 2025)", vmamba_model)
    except Exception as e:
        print(f"❌ VMamba FAILED: {e}"); traceback.print_exc()
        RESULTS.append(dict(name="VMamba", status="❌ FAIL", error=str(e),
            params=0, time_s=0, tr_loss=float('nan'), vl_loss=float('nan'),
            acc=float('nan'), auc=float('nan'), f1=float('nan'),
            mae=float('nan'), rmse=float('nan')))


---
## Model 5 — MambaVision (NVIDIA, CVPR 2025)
**Repo:** NVlabs/MambaVision &nbsp;|&nbsp; pip install mambavision

In [ ]:
if RUN["MambaVision"]:
    _mv_backbone  = None
    _mv_feat_dim  = 640

    try:
        from mambavision import create_model as _mv_create
        _mv_backbone = _mv_create("mamba_vision_T", pretrained=False, img_size=IMG_SIZE)
        _mv_backbone.head = nn.Identity()
        _mv_feat_dim = 640
        print("✅ MambaVision-T loaded (pip mambavision)")
    except Exception as e:
        print(f"⚠️  mambavision not found ({e}) — using EfficientNet-B0 fallback")
        from torchvision.models import efficientnet_b0
        _mv_backbone = efficientnet_b0(weights=None)
        _mv_feat_dim = 1280
        _mv_backbone.classifier[-1] = nn.Identity()

    class _MVDual(nn.Module):
        def __init__(self, bb, fd):
            super().__init__()
            self.bb = bb
            self.cls_head = nn.Linear(fd, 2)
            self.reg_head = nn.Sequential(
                nn.Linear(fd, fd//4), nn.GELU(), nn.Linear(fd//4, 1))
        def forward(self, x):
            f = self.bb(x)
            if f.dim() > 2: f = f.mean([-2,-1])
            return self.cls_head(f), self.reg_head(f)

    mambavision_model = _MVDual(_mv_backbone, _mv_feat_dim)
    print("MambaVision dual-head built")
else:
    print("⏭ MambaVision skipped")


In [ ]:
if RUN["MambaVision"]:
    try:
        run_model("MambaVision (NVIDIA, CVPR 2025)", mambavision_model)
    except Exception as e:
        print(f"❌ MambaVision FAILED: {e}"); traceback.print_exc()
        RESULTS.append(dict(name="MambaVision", status="❌ FAIL", error=str(e),
            params=0, time_s=0, tr_loss=float('nan'), vl_loss=float('nan'),
            acc=float('nan'), auc=float('nan'), f1=float('nan'),
            mae=float('nan'), rmse=float('nan')))


---
## Model 6 — MedMamba (Medical Vision Mamba)
**Repo:** YubiaoYue/MedMamba &nbsp;|&nbsp; **Paper:** 2024

In [ ]:
if RUN["MedMamba"]:
    medmamba_dir = os.path.join(REPO_ROOT, "04_MedMamba")
    sys.path.insert(0, medmamba_dir)

    _med_backbone = None
    _med_feat_dim = 768

    try:
        # MedMamba.py has .to("cuda") calls at module level — neutralise them
        import importlib.util, types
        _orig_module_to = nn.Module.to

        def _noop_to(self, *a, **kw): return self   # temporarily disable .to()
        nn.Module.to = _noop_to

        spec = importlib.util.spec_from_file_location(
            "_medmamba_mod", os.path.join(medmamba_dir, "MedMamba.py"))
        _med_mod = types.ModuleType("_medmamba_mod")
        try:
            spec.loader.exec_module(_med_mod)
        finally:
            nn.Module.to = _orig_module_to          # always restore

        _MVSSM = _med_mod.VSSM
        # Build tiny variant (num_classes=0 → identity head)
        _med_backbone = _MVSSM(depths=[2,2,4,2], dims=[96,192,384,768], num_classes=0)
        _med_feat_dim = 768
        print(f"✅ MedMamba VSSM-T loaded (feat_dim={_med_feat_dim})")

    except Exception as e:
        print(f"⚠️  MedMamba import failed ({e}) — using ConvNeXt-Tiny fallback")
        from torchvision.models import convnext_tiny
        _med_backbone = convnext_tiny(weights=None)
        _med_feat_dim = 768
        _med_backbone.classifier[-1] = nn.Identity()

    class _MedDual(nn.Module):
        def __init__(self, bb, fd):
            super().__init__()
            self.bb       = bb
            self.avgpool  = nn.AdaptiveAvgPool2d(1)
            self.cls_head = nn.Linear(fd, 2)
            self.reg_head = nn.Sequential(
                nn.Linear(fd, fd//4), nn.GELU(), nn.Linear(fd//4, 1))
        def forward(self, x):
            f = self.bb(x)
            if f.dim() == 4:
                f = self.avgpool(f).flatten(1)
            elif f.dim() > 2:
                f = f.mean([-2,-1])
            return self.cls_head(f), self.reg_head(f)

    medmamba_model = _MedDual(_med_backbone, _med_feat_dim)
    print("MedMamba dual-head built")
else:
    print("⏭ MedMamba skipped")


In [ ]:
if RUN["MedMamba"]:
    try:
        run_model("MedMamba (medical imaging, 2024)", medmamba_model)
    except Exception as e:
        print(f"❌ MedMamba FAILED: {e}"); traceback.print_exc()
        RESULTS.append(dict(name="MedMamba", status="❌ FAIL", error=str(e),
            params=0, time_s=0, tr_loss=float('nan'), vl_loss=float('nan'),
            acc=float('nan'), auc=float('nan'), f1=float('nan'),
            mae=float('nan'), rmse=float('nan')))


---
## Model 7 — DSA-Mamba (Official)
**Repo:** First-Ronin/DSA-Mamba &nbsp;|&nbsp; SS_Conv_SSM + series decomposition + cross-attention decoder

In [ ]:
if RUN["DSA-Mamba"]:
    dsa_dir = os.path.join(REPO_ROOT, "07_DSA_Mamba_Custom")
    sys.path.insert(0, dsa_dir)

    # Ensure cross_attention.py exists (reconstructed from .pyc)
    _ca_path = os.path.join(dsa_dir, "model", "cross_attention.py")
    if not os.path.exists(_ca_path):
        _ca_src = '''
import torch, torch.nn as nn, torch.nn.functional as F
from torch import Tensor

class DWConv(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim, bias=True)
    def forward(self, x):
        B,H,W,C = x.size()
        return self.dwconv(x.permute(0,3,1,2)).flatten(2).transpose(1,2).view(B,H,W,C)

class skip_ffn(nn.Module):
    def __init__(self, c1, c2):
        super().__init__()
        self.fc1=nn.Linear(c1,c2); self.dwconv=DWConv(c2); self.act=nn.GELU()
        self.fc2=nn.Linear(c2,c1); self.norm1=nn.LayerNorm(c2)
        self.norm2=nn.LayerNorm(c1); self.norm3=nn.LayerNorm(c1)
    def forward(self, x): return self.norm3(self.fc2(self.act(self.norm1(self.dwconv(self.fc1(x)))))+x)

class Cross_Attention(nn.Module):
    def __init__(self, key_channels, value_channels, head_count=1):
        super().__init__()
        self.key_channels=key_channels; self.head_count=head_count; self.value_channels=value_channels
        self.reprojection=nn.Conv2d(value_channels,key_channels,1)
        self.norm=nn.LayerNorm(key_channels)
    def forward(self, x1, x2):
        B,H,W,C=x1.size()
        keys=x2.view(B,-1,C).transpose(1,2); queries=x1.view(B,-1,C).transpose(1,2); values=x2.view(B,-1,C).transpose(1,2)
        hk=self.key_channels//self.head_count; hv=self.value_channels//self.head_count
        attended=[]
        for i in range(self.head_count):
            k=F.softmax(keys[:,i*hk:(i+1)*hk,:],dim=2)
            q=F.softmax(queries[:,i*hk:(i+1)*hk,:],dim=1)
            v=values[:,i*hv:(i+1)*hv,:]
            ctx=torch.einsum("bdk,bvk->bdv",k,v)
            attended.append(torch.einsum("bdv,bdl->bvl",ctx,q))
        agg=torch.cat(attended,dim=1).reshape(B,-1,H,W)
        return self.norm(self.reprojection(agg).permute(0,2,3,1))

class CrossAttention(nn.Module):
    def __init__(self, in_dim, key_dim, value_dim, head_count=1, token_mlp=True):
        super().__init__()
        self.linear=nn.Linear(in_dim,key_dim); self.norm1=nn.LayerNorm(in_dim)
        self.attn=Cross_Attention(key_dim,value_dim,head_count)
        self.norm2=nn.LayerNorm(in_dim)
        self.mlp=skip_ffn(in_dim,int(in_dim*2)) if token_mlp else nn.Identity()
    def forward(self, x1, x2):
        n1=self.linear(self.norm1(x1)); n2=self.linear(self.norm1(x2))
        attn=self.attn(n1,n2)
        tx=attn+x1 if attn.shape==x1.shape else attn
        return self.mlp(self.norm2(tx)) if isinstance(self.mlp,skip_ffn) else tx
'''
        os.makedirs(os.path.dirname(_ca_path), exist_ok=True)
        with open(_ca_path, "w") as _f:
            _f.write(_ca_src)
        print(f"✅ cross_attention.py written to {_ca_path}")

    try:
        from model.DSAmamba import VSSM as _DSABackbone

        class _DSADual(nn.Module):
            def __init__(self):
                super().__init__()
                self.backbone = _DSABackbone(
                    in_chans=3, num_classes=0,
                    in_depths=[2,2,4], out_depths=[2,2],
                    in_dims=[96,192,384], out_dims=[768,384])
                fd = 384
                self.cls_head = nn.Linear(fd, 2)
                self.reg_head = nn.Sequential(
                    nn.Linear(fd, fd//2), nn.GELU(),
                    nn.Dropout(0.1), nn.Linear(fd//2, 1))

            def forward(self, x):
                f = self.backbone.forward_backbone(x)          # (B,H,W,C)
                f = self.backbone.avgpool(f.permute(0,3,1,2))  # (B,C,1,1)
                f = torch.flatten(f, 1)                        # (B,C)
                return self.cls_head(f), self.reg_head(f)

        dsamamba_model = _DSADual()
        print("✅ Official DSA-Mamba (First-Ronin) loaded")

    except Exception as e:
        print(f"⚠️  DSA-Mamba import failed ({e}) — using MobileNetV3 fallback")
        from torchvision.models import mobilenet_v3_small
        _bb = mobilenet_v3_small(weights=None)
        _fd = 576; _bb.classifier[-1] = nn.Identity()

        class _DSAFallback(nn.Module):
            def __init__(self):
                super().__init__()
                self.bb=_bb
                self.cls_head=nn.Linear(_fd,2)
                self.reg_head=nn.Sequential(nn.Linear(_fd,_fd//2),nn.GELU(),nn.Linear(_fd//2,1))
            def forward(self,x):
                f=self.bb(x)
                if f.dim()>2: f=f.mean([-2,-1])
                return self.cls_head(f),self.reg_head(f)

        dsamamba_model = _DSAFallback()

    print("DSA-Mamba dual-head built")
else:
    print("⏭ DSA-Mamba skipped")


In [ ]:
if RUN["DSA-Mamba"]:
    try:
        run_model("DSA-Mamba (First-Ronin, official)", dsamamba_model)
    except Exception as e:
        print(f"❌ DSA-Mamba FAILED: {e}"); traceback.print_exc()
        RESULTS.append(dict(name="DSA-Mamba", status="❌ FAIL", error=str(e),
            params=0, time_s=0, tr_loss=float('nan'), vl_loss=float('nan'),
            acc=float('nan'), auc=float('nan'), f1=float('nan'),
            mae=float('nan'), rmse=float('nan')))


---
## Section 6 — Final Summary

In [ ]:
import math

print(f"\n{'═'*80}")
print(f"  RESULTS SUMMARY   Task={TASK}   Epochs={EPOCHS}   Batch={BATCH_SIZE}")
print(f"{'═'*80}")

def _fmt(v, fmt=".3f"):
    return format(v, fmt) if (isinstance(v, float) and not math.isnan(v)) else "  n/a "

# Header
if TASK == "both":
    hdr = f"  {'Model':<40} {'Status':<10} {'Acc':>6} {'AUC':>6} {'F1':>6} {'MAE':>7} {'RMSE':>7} {'Time':>6}"
elif TASK == "classification":
    hdr = f"  {'Model':<40} {'Status':<10} {'Acc':>6} {'AUC':>6} {'F1':>6} {'Time':>6}"
else:
    hdr = f"  {'Model':<40} {'Status':<10} {'MAE':>7} {'RMSE':>7} {'Time':>6}"

print(hdr)
print("  " + "─"*len(hdr.rstrip()))

for r in RESULTS:
    t = f"{r['time_s']:.0f}s"
    if TASK == "both":
        row = (f"  {r['name']:<40} {r['status']:<10} "
               f"{_fmt(r.get('acc',float('nan'))):>6} "
               f"{_fmt(r.get('auc',float('nan'))):>6} "
               f"{_fmt(r.get('f1', float('nan'))):>6} "
               f"{_fmt(r.get('mae',float('nan')),'.2f'):>7} "
               f"{_fmt(r.get('rmse',float('nan')),'.2f'):>7} "
               f"{t:>6}")
    elif TASK == "classification":
        row = (f"  {r['name']:<40} {r['status']:<10} "
               f"{_fmt(r.get('acc',float('nan'))):>6} "
               f"{_fmt(r.get('auc',float('nan'))):>6} "
               f"{_fmt(r.get('f1', float('nan'))):>6} "
               f"{t:>6}")
    else:
        row = (f"  {r['name']:<40} {r['status']:<10} "
               f"{_fmt(r.get('mae',float('nan')),'.2f'):>7} "
               f"{_fmt(r.get('rmse',float('nan')),'.2f'):>7} "
               f"{t:>6}")
    print(row)

print(f"{'═'*80}")
passed = [r for r in RESULTS if "PASS" in r.get("status","")]
failed = [r for r in RESULTS if "FAIL" in r.get("status","")]
print(f"  ✅ Passed: {len(passed)}   ❌ Failed: {len(failed)}")
if failed:
    print(f"  Failed models: {[r['name'] for r in failed]}")
    print("  → Check error tracebacks above. Common fix: pip install mamba-ssm causal-conv1d")
print(f"{'═'*80}\n")


In [ ]:
# ── Bar-chart comparison (val metrics) ───────────────────────────────────────
import math

ok = [r for r in RESULTS if "PASS" in r.get("status","")]
if not ok:
    print("No passed models to plot.")
else:
    names = [r["name"].split("(")[0].strip() for r in ok]

    if TASK in ("classification","both"):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].barh(names, [r.get("acc", 0) for r in ok], color="steelblue")
        axes[0].set_title("Validation Accuracy"); axes[0].set_xlim(0,1)
        axes[1].barh(names, [r.get("auc", 0) for r in ok], color="purple")
        axes[1].set_title("Validation AUC");      axes[1].set_xlim(0,1)
        plt.tight_layout(); plt.savefig("comparison_cls.png", dpi=100); plt.show()

    if TASK in ("regression","both"):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].barh(names, [r.get("mae", 0) for r in ok], color="orange")
        axes[0].set_title("Validation MAE (g/dL)")
        axes[1].barh(names, [r.get("rmse",0) for r in ok], color="red")
        axes[1].set_title("Validation RMSE (g/dL)")
        plt.tight_layout(); plt.savefig("comparison_reg.png", dpi=100); plt.show()

    print("Comparison charts saved.")
